# Unibet Over/Under (Kambi)
Notebook pour tester la recuperation des marches Over/Under (Total Points) NBA via l'API Kambi.
Affiche les cotes pour comparer avec l'interface Unibet FR et expose la brand/endpoint utilises.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


In [2]:
DAYS_AHEAD = 0  # 0 = aujourd'hui, 1 = demain, etc.
LOCAL_TZ = "Europe/Paris"
FALLBACK_NEXT = True
DEBUG_PROBE_BRANDS = True  # True pour tracer les essais base/brand


In [3]:
from odds.over_under.unibet import fetch_nba_totals_for_day

try:
    markets = await fetch_nba_totals_for_day(
        days_ahead=DAYS_AHEAD,
        tz_name=LOCAL_TZ,
        fallback_to_next_available=FALLBACK_NEXT,
        debug_probe_brands=DEBUG_PROBE_BRANDS,
    )
    brands = sorted({m.brand for m in markets if m.brand})
    base_urls = sorted({m.base_url for m in markets if m.base_url})
    print(f"Recupere {len(markets)} marches. Brands: {brands or 'n/a'}")
    if base_urls:
        print(f"Endpoints: {base_urls}")
except Exception as exc:
    markets = []
    print(f"Erreur pendant la recuperation: {exc}")
len(markets)


[probe] base=eu-offering-api.kambicdn.com/offering/v2018 brand=unibetfrubfr path=basketball/nba/all/all/matches.json status=http_429 events=0 error=Client error '429 Too Many Requests' for url 'https://eu-offering-api.kambicdn.com/offering/v2018/unibetfrubfr/listView/basketball/nba/all/all/matches.json?lang=fr_FR&market=FR&client_id=2&channel_id=1&useCombined=true&includeParticipants=true'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429
[probe] base=eu-offering-api.kambicdn.com/offering/v2018 brand=unibetfrubfr path=basketball/nba/all/matches.json status=http_429 events=0 error=Client error '429 Too Many Requests' for url 'https://eu-offering-api.kambicdn.com/offering/v2018/unibetfrubfr/listView/basketball/nba/all/matches.json?lang=fr_FR&market=FR&client_id=2&channel_id=1&useCombined=true&includeParticipants=true'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429
[probe] base=eu-offering-api.kambicdn.com/of

105

In [4]:
import pandas as pd

if not markets:
    print("Aucun marche recupere.")
    df_markets = pd.DataFrame()
else:
    def markets_to_frame(snapshots):
        rows = []
        for market in snapshots:
            for sel in market.selections:
                line_raw = sel.raw.get("line") if isinstance(sel.raw, dict) else None
                pivot = None
                if isinstance(line_raw, (int, float)):
                    pivot = round(line_raw / 1000.0, 3)
                direction_raw = str(sel.raw.get("type", sel.name or "")).lower()
                if any(tok in direction_raw for tok in ["over", "plus"]):
                    direction = "over"
                elif any(tok in direction_raw for tok in ["under", "moins"]):
                    direction = "under"
                else:
                    direction = direction_raw
                rows.append({
                    "event_id": market.event_id,
                    "event_name": market.event_name,
                    "competition": market.competition,
                    "market_id": market.market_id,
                    "market_name": market.market_name,
                    "selection": sel.name,
                    "direction": direction,
                    "pivot": pivot,
                    "odds_decimal": sel.odds_decimal,
                    "fractional": sel.odds_fractional,
                    "brand": market.brand,
                    "base_url": market.base_url,
                    "start_utc": market.start_utc,
                    "local_start": market.local_start,
                    "captured_at": market.captured_at,
                })
        return pd.DataFrame(rows)

    df_markets = markets_to_frame(markets)
    df_markets.sort_values(
        ["start_utc", "event_name", "market_id", "pivot", "direction"],
        inplace=True,
    )

df_markets.head(12)


,event_id,event_name,competition,market_id,market_name,selection,direction,pivot,odds_decimal,fractional,brand,base_url,start_utc,local_start,captured_at
0,1024652750,Cleveland Cavaliers - Memphis Grizzlies,NBA,2577104582,Total de points - Prolongations incluses,Plus de,over,238.0,1.727,8/11,ubuk,https://eu-offering-api.kambicdn.com/offering/...,2025-11-15 22:00:00+00:00,2025-11-15 23:00:00+01:00,2025-11-15 20:49:08.497903+00:00
1,1024652750,Cleveland Cavaliers - Memphis Grizzlies,NBA,2577104582,Total de points - Prolongations incluses,Moins de,under,238.0,2.100,11/10,ubuk,https://eu-offering-api.kambicdn.com/offering/...,2025-11-15 22:00:00+00:00,2025-11-15 23:00:00+01:00,2025-11-15 20:49:08.497903+00:00
2,1024652750,Cleveland Cavaliers - Memphis Grizzlies,NBA,2577131707,Total de points - Prolongations incluses,Plus de,over,237.5,1.700,7/10,ubuk,https://eu-offering-api.kambicdn.com/offering/...,2025-11-15 22:00:00+00:00,2025-11-15 23:00:00+01:00,2025-11-15 20:49:08.497972+00:00
3,1024652750,Cleveland Cavaliers - Memphis Grizzlies,NBA,2577131707,Total de points - Prolongations incluses,Moins de,under,237.5,2.150,23/20,ubuk,https://eu-offering-api.kambicdn.com/offering/...,2025-11-15 22:00:00+00:00,2025-11-15 23:00:00+01:00,2025-11-15 20:49:08.497972+00:00
4,1024652750,Cleveland Cavaliers - Memphis Grizzlies,NBA,2577173953,Total de points - Prolongations incluses,Plus de,over,238.5,1.750,3/4,ubuk,https://eu-offering-api.kambicdn.com/offering/...,2025-11-15 22:00:00+00:00,2025-11-15 23:00:00+01:00,2025-11-15 20:49:08.498015+00:00
5,1024652750,Cleveland Cavaliers - Memphis Grizzlies,NBA,2577173953,Total de points - Prolongations incluses,Moins de,under,238.5,2.050,21/20,ubuk,https://eu-offering-api.kambicdn.com/offering/...,2025-11-15 22:00:00+00:00,2025-11-15 23:00:00+01:00,2025-11-15 20:49:08.498015+00:00
6,1024652750,Cleveland Cavaliers - Memphis Grizzlies,NBA,2577174681,Total de points - Prolongations incluses,Plus de,over,245.0,2.200,6/5,ubuk,https://eu-offering-api.kambicdn.com/offering/...,2025-11-15 22:00:00+00:00,2025-11-15 23:00:00+01:00,2025-11-15 20:49:08.498043+00:00
7,1024652750,Cleveland Cavaliers - Memphis Grizzlies,NBA,2577174681,Total de points - Prolongations incluses,Moins de,under,245.0,1.650,13/20,ubuk,https://eu-offering-api.kambicdn.com/offering/...,2025-11-15 22:00:00+00:00,2025-11-15 23:00:00+01:00,2025-11-15 20:49:08.498043+00:00
8,1024652750,Cleveland Cavaliers - Memphis Grizzlies,NBA,2577174682,Total de points - Prolongations incluses,Plus de,over,230.0,1.350,7/20,ubuk,https://eu-offering-api.kambicdn.com/offering/...,2025-11-15 22:00:00+00:00,2025-11-15 23:00:00+01:00,2025-11-15 20:49:08.498071+00:00
9,1024652750,Cleveland Cavaliers - Memphis Grizzlies,NBA,2577174682,Total de points - Prolongations incluses,Moins de,under,230.0,3.000,2/1,ubuk,https://eu-offering-api.kambicdn.com/offering/...,2025-11-15 22:00:00+00:00,2025-11-15 23:00:00+01:00,2025-11-15 20:49:08.498071+00:00


In [5]:
if not df_markets.empty:
    print("Resume nb marchés par match:")
    display(
        df_markets.groupby("event_name")["market_id"]
        .nunique()
        .sort_values(ascending=False)
        .to_frame("nb_markets")
    )

    # Tableau pivoté Over/Under ordonné par pivot croissant
    df_tidy = df_markets.copy()
    table = (
        df_tidy.pivot_table(
            index=["event_name", "market_id", "pivot"],
            columns="direction",
            values="odds_decimal",
            aggfunc="first",
        )
        .reset_index()
    )
    table = table.rename(columns={"over": "odd_over", "under": "odd_under"})
    table = table.sort_values(["event_name", "pivot"])
    display(table[["event_name", "pivot", "odd_under", "odd_over"]])
else:
    print("Pas de table a afficher.")


Resume nb marchés par match:


,nb_markets
event_name,
Cleveland Cavaliers - Memphis Grizzlies,105


direction,event_name,pivot,odd_under,odd_over
91,Cleveland Cavaliers - Memphis Grizzlies,29.5,1.769,1.950
84,Cleveland Cavaliers - Memphis Grizzlies,32.5,1.920,1.800
93,Cleveland Cavaliers - Memphis Grizzlies,57.5,1.800,1.920
92,Cleveland Cavaliers - Memphis Grizzlies,58.5,1.800,1.909
77,Cleveland Cavaliers - Memphis Grizzlies,59.5,1.850,1.850
88,Cleveland Cavaliers - Memphis Grizzlies,59.5,2.300,1.533
70,Cleveland Cavaliers - Memphis Grizzlies,60.5,2.100,1.650
73,Cleveland Cavaliers - Memphis Grizzlies,61.5,1.950,1.800
100,Cleveland Cavaliers - Memphis Grizzlies,61.5,1.820,1.909
69,Cleveland Cavaliers - Memphis Grizzlies,62.5,1.769,1.950


In [6]:
import json

if markets:
    snap = markets[0]
    print(f"{snap.event_name} | {snap.market_name}")
    print(f"Start local: {snap.local_start} // start utc: {snap.start_utc}")
    print(f"Brand: {snap.brand} | Endpoint: {snap.base_url}")
    print(f"Selections: {len(snap.selections)}")
    display(pd.json_normalize([sel.to_dict() for sel in snap.selections]))
    print("Offre brute (tronquee):")
    print(json.dumps(snap.raw, indent=2)[:4000])
else:
    print("Pas de marche a inspecter.")


Cleveland Cavaliers - Memphis Grizzlies | Total de points - Prolongations incluses
Start local: 2025-11-15 23:00:00+01:00 // start utc: 2025-11-15 22:00:00+00:00
Brand: ubuk | Endpoint: https://eu-offering-api.kambicdn.com/offering/v2018
Selections: 2


,name,odds_decimal,odds_fractional,popularity,true_probability_or,true_odds_or,true_probability_mpo,true_odds_mpo,book_overround,raw.id,raw.label,raw.englishLabel,raw.odds,raw.line,raw.type,raw.betOfferId,raw.changedDate,raw.oddsFractional,raw.oddsAmerican,raw.status,raw.cashOutStatus
0,Plus de,1.727,8/11,None,0.548733,1.822381,0.554116,1.804677,0.055229,3953652988,Plus de,Over,1730,238000,OT_OVER,2577104582,2025-11-15T20:24:47Z,8/11,-137,OPEN,ENABLED
1,Moins de,2.100,11/10,None,0.451267,2.215981,0.445884,2.242734,0.055229,3953652989,Moins de,Under,2120,238000,OT_UNDER,2577104582,2025-11-15T20:24:47Z,11/10,112,OPEN,ENABLED


Offre brute (tronquee):
{
  "id": 2577104582,
  "closed": "2025-11-15T22:00:00Z",
  "criterion": {
    "id": 1001159509,
    "label": "Total de points - Prolongations incluses",
    "englishLabel": "Total Points - Including Overtime",
    "order": [],
    "occurrenceType": "POINTS",
    "lifetime": "FULL_TIME_OVERTIME"
  },
  "betOfferType": {
    "id": 6,
    "name": "Plus de/Moins de",
    "englishName": "Over/Under"
  },
  "eventId": 1024652750,
  "outcomes": [
    {
      "id": 3953652988,
      "label": "Plus de",
      "englishLabel": "Over",
      "odds": 1730,
      "line": 238000,
      "type": "OT_OVER",
      "betOfferId": 2577104582,
      "changedDate": "2025-11-15T20:24:47Z",
      "oddsFractional": "8/11",
      "oddsAmerican": "-137",
      "status": "OPEN",
      "cashOutStatus": "ENABLED"
    },
    {
      "id": 3953652989,
      "label": "Moins de",
      "englishLabel": "Under",
      "odds": 2120,
      "line": 238000,
      "type": "OT_UNDER",
      "betOfferId":